# 02 — Boîte à outils : septembre contre octobre

**IA-01 · Jour 4 · Hexalp · Matinée en autonomie**

Ce notebook n'est pas un parcours guidé. C'est une **boîte à outils** : des cellules prêtes à l'emploi pour comparer les deux mois, adapter la préparation et évaluer votre modèle sur des données qu'il n'a jamais vues. Prenez celles dont vous avez besoin, dans l'ordre qui vous convient, et reportez vos constats dans votre carnet.

Une seule cellule contient un trou `___` : celle de la section 2. À vous de la compléter.

| Section | Outil | Séquence du kit |
|---|---|---|
| 0 | Vos paramètres du J3 | 4.2 |
| 1 | Comparer septembre et octobre | 4.2 |
| 2 | Adapter la préparation | 4.2 |
| 3 | Le « 99 % » de Sofia | 4.2 |
| 4 | Entraîner sur septembre, appliquer à octobre | 4.3 |
| 5 | Mesurer | 4.3 |
| 6 | Regarder les erreurs | 4.3 |
| 7 | Bonus : réentraîner | 4.3 |

⚠️ Le modèle ne bloque rien. Il produit des alertes qu'un analyste doit qualifier.

---
## 0. Vos paramètres

Reprenez **exactement** les choix de votre script `detection_ssh.py` du J3. Les valeurs ci-dessous sont celles du corrigé : remplacez-les si les vôtres sont différentes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

FICHIER_SEPT = "hexalp_ssh_sept2026.csv"
FICHIER_OCT = "hexalp_ssh_oct2026.csv"
GRAINE = 42

CARACTERISTIQUES = ["echecs_auth", "succes_auth", "comptes_distincts", "duree_session_min",
                    "octets_sortants_mo", "heure", "externe"]
PROFONDEUR_ARBRE = 3
CONTAMINATION = 0.05

---
## 1. Comparer septembre et octobre

Avant d'appliquer un modèle à de nouvelles données, on vérifie qu'elles ressemblent à celles qui ont servi à l'entraîner. Attention : `oct` est un mot réservé de Python, on écrit donc `octobre` en entier.

In [ ]:
septembre = pd.read_csv(FICHIER_SEPT)
octobre = pd.read_csv(FICHIER_OCT)

print("Septembre :", septembre.shape, "| Octobre :", octobre.shape)
print("Colonnes apparues en octobre  :", set(octobre.columns) - set(septembre.columns))
print("Colonnes disparues en octobre :", set(septembre.columns) - set(octobre.columns))

In [ ]:
# Les cases vides, colonne par colonne
pd.DataFrame({"septembre": septembre.isna().sum(), "octobre": octobre.isna().sum()})

In [ ]:
# Les colonnes de volume, jour par jour : combien de cases remplies ?
octobre.groupby("date")[["octets_sortants_mo", "octets_sortants_ko"]].count()

In [ ]:
# Les statistiques des deux mois, côte à côte
stats = {"septembre": septembre.describe().T[["mean", "50%", "max"]],
         "octobre": octobre.describe().T[["mean", "50%", "max"]]}
pd.concat(stats, axis=1).round(1)

In [ ]:
# La part d'attaques et la répartition interne / externe
print(septembre["etiquette"].value_counts(normalize=True).round(3))
print(octobre["etiquette"].value_counts(normalize=True).round(3))
print(pd.crosstab(octobre["origine"], octobre["etiquette"]))

In [ ]:
# Les adresses internes jamais vues en septembre
nouvelles = octobre[(octobre["origine"] == "interne") & ~octobre["ip_source"].isin(septembre["ip_source"])]
nouvelles.groupby(["ip_source", "etiquette"]).size().sort_values(ascending=False)

In [ ]:
# Le nombre d'attaques étiquetées par Hugo, jour par jour
octobre[octobre["etiquette"] == "attaque"].groupby("date").size().plot(kind="bar", figsize=(9, 3), title="Attaques par jour en octobre")
plt.show()

---
## 2. Adapter la préparation

Croisez ce que vous venez de voir avec le **journal des changements** de la DSI (kit, section 4.2). Puis complétez la cellule : la colonne `octets_sortants_mo` doit contenir des **mégaoctets** pour toutes les lignes d'octobre, comme en septembre.

💡 1 Mo = 1 024 Ko.

In [ ]:
# 1. Octobre seulement : là où l'ancienne sonde n'a rien mesuré, on reprend la mesure de la nouvelle sonde, convertie en Mo
octobre["octets_sortants_mo"] = octobre["octets_sortants_mo"].fillna(___)

# 2. Les deux mois, comme au J3 : les cases encore vides (pannes de sonde) prennent la médiane
septembre["octets_sortants_mo"] = septembre["octets_sortants_mo"].fillna(septembre["octets_sortants_mo"].median())
octobre["octets_sortants_mo"] = octobre["octets_sortants_mo"].fillna(octobre["octets_sortants_mo"].median())

# 3. La colonne texte origine devient un nombre
septembre["externe"] = (septembre["origine"] == "externe").astype(int)
octobre["externe"] = (octobre["origine"] == "externe").astype(int)

print("Cases vides restantes en octobre :", octobre[CARACTERISTIQUES].isna().sum().sum())
octobre[["date", "heure", "ip_source", "octets_sortants_mo", "octets_sortants_ko"]].sample(5, random_state=GRAINE)

---
## 3. Le « 99 % » de Sofia

Sofia a simplement changé le nom du fichier dans votre script du J3 et l'a relancé. Cette cellule refait exactement la même chose : elle **découpe octobre** en entraînement et test, puis évalue.

In [ ]:
X_o = octobre[CARACTERISTIQUES]
y_o = (octobre["etiquette"] == "attaque").astype(int)
Xa, Xb, ya, yb = train_test_split(X_o, y_o, test_size=0.3, stratify=y_o, random_state=GRAINE)
arbre_sofia = DecisionTreeClassifier(max_depth=PROFONDEUR_ARBRE, random_state=GRAINE).fit(Xa, ya)
print("Accuracy :", round(accuracy_score(yb, arbre_sofia.predict(Xb)), 3))

Pour obtenir ce chiffre, le modèle a été **entraîné avec les étiquettes d'octobre**. Le lundi matin, quand le script tourne sur la semaine écoulée, ces étiquettes existent-elles déjà ? Répondez dans votre carnet (séquence 4.2).

---
## 4. Entraîner sur septembre, appliquer à octobre

C'est la situation réelle : le modèle validé au J3 apprend sur septembre, puis on lui présente octobre sans lui montrer les étiquettes. Les étiquettes de Hugo ne servent qu'à **vérifier** ses réponses.

L'arbre est entraîné comme au J3, sur 70 % de septembre : c'est **le même arbre** que celui que vous avez présenté à Nadia.

In [ ]:
X_s = septembre[CARACTERISTIQUES]
y_s = (septembre["etiquette"] == "attaque").astype(int)

# L'arbre du J3 : mêmes données, même découpage, même graine
X_train, X_test, y_train, y_test = train_test_split(X_s, y_s, test_size=0.3, stratify=y_s, random_state=GRAINE)
arbre = DecisionTreeClassifier(max_depth=PROFONDEUR_ARBRE, random_state=GRAINE).fit(X_train, y_train)
print(export_text(arbre, feature_names=CARACTERISTIQUES))

# L'Isolation Forest du J3 : tout septembre, sans étiquettes
iforest = IsolationForest(contamination=CONTAMINATION, random_state=GRAINE).fit(X_s)

# On applique les deux modèles à octobre
octobre["avis_arbre"] = arbre.predict(X_o)
octobre["score_anomalie"] = iforest.decision_function(X_o)
octobre["anomalie"] = (iforest.predict(X_o) == -1).astype(int)
octobre["alerte"] = ((octobre["avis_arbre"] == 1) | (octobre["anomalie"] == 1)).astype(int)
octobre["alerte"].value_counts()

---
## 5. Mesurer

Trois résultats à comparer : l'arbre seul, l'Isolation Forest seul, et l'alerte finale de votre script (l'un **ou** l'autre).

In [ ]:
for nom, colonne in [("Arbre de décision", "avis_arbre"), ("Isolation Forest", "anomalie"), ("Alerte finale", "alerte"), ("fail2ban", "fail2ban_ban")]:
    tn, fp, fn, tp = confusion_matrix(y_o, octobre[colonne]).ravel()
    print(f"=== {nom} : {fp} faux positifs, {fn} faux négatifs, {tp} attaques détectées")
    print(classification_report(y_o, octobre[colonne], target_names=["normal", "attaque"], digits=2))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_o, octobre["alerte"], display_labels=["normal", "attaque"], cmap="Blues")
plt.title("Alerte finale — octobre")
plt.show()

In [ ]:
# Le nombre d'alertes par semaine (l'astreinte peut en traiter 30)
octobre["semaine"] = pd.to_datetime(octobre["date"]).dt.isocalendar().week
octobre.groupby("semaine")["alerte"].sum()

---
## 6. Regarder les erreurs

Un taux ne dit pas **quelles** erreurs le modèle commet. Ces cellules les listent.

In [ ]:
colonnes = ["date", "heure", "ip_source", "origine", "echecs_auth", "succes_auth", "comptes_distincts",
            "duree_session_min", "octets_sortants_mo", "avis_arbre", "anomalie", "score_anomalie"]

# Les faux positifs, regroupés par adresse
faux_positifs = octobre[(octobre["alerte"] == 1) & (octobre["etiquette"] == "normal")]
faux_positifs.groupby("ip_source").size().sort_values(ascending=False)

In [ ]:
faux_positifs[colonnes]

In [ ]:
# Les faux négatifs : à quoi ressemblent les attaques ratées ?
faux_negatifs = octobre[(octobre["alerte"] == 0) & (octobre["etiquette"] == "attaque")]
print(len(faux_negatifs), "attaques ratées")
faux_negatifs[colonnes].describe().round(1)

In [ ]:
# Regroupées par profil : origine, comptes testés, succès
faux_negatifs.groupby(["origine", "comptes_distincts", "succes_auth"]).size()

In [ ]:
faux_negatifs[colonnes].sort_values(["date", "heure"]).head(30)

In [ ]:
# Exporter les alertes d'octobre pour l'analyste, les plus inhabituelles d'abord
alertes = octobre[octobre["alerte"] == 1].sort_values("score_anomalie")
alertes[colonnes + ["etiquette"]].to_csv("alertes_octobre.csv", index=False)
print(len(alertes), "alertes écrites dans alertes_octobre.csv")

---
## 7. Bonus : réentraîner

Hugo a étiqueté octobre. Et si l'on réentraînait l'arbre sur **septembre + la première semaine d'octobre**, puis qu'on le testait sur la **seconde semaine** ?

In [ ]:
semaine1 = octobre["date"] < "2026-10-05"
semaine2 = ~semaine1

X_nouveau = pd.concat([X_s, X_o[semaine1]])
y_nouveau = pd.concat([y_s, y_o[semaine1]])

for nom, modele in [("Arbre du J3", arbre),
                    ("Arbre réentraîné", DecisionTreeClassifier(max_depth=PROFONDEUR_ARBRE, random_state=GRAINE).fit(X_nouveau, y_nouveau))]:
    p = modele.predict(X_o[semaine2])
    tn, fp, fn, tp = confusion_matrix(y_o[semaine2], p).ravel()
    print(f"{nom:18s} → semaine 2 : {fp} faux positifs, {fn} faux négatifs")

print(export_text(DecisionTreeClassifier(max_depth=PROFONDEUR_ARBRE, random_state=GRAINE).fit(X_nouveau, y_nouveau), feature_names=CARACTERISTIQUES))

✅ Vous avez tout ce qu'il faut pour écrire `detection_ssh_v2.py` (séquence 4.3) et votre rapport d'évaluation (séquence 4.4).